# Laboratorio 3 · RAG del copiloto BancAnalytics
**Rama:** `l3_DSC_30ME`. Continuación de `laboratorio-02.ipynb`.

Caso: copiloto de asesores de un banco **100 % ficticio**. Conserva su rol,
restricciones de alcance, memoria de corto plazo y compactación por resúmenes.
Ahora responde condiciones de productos usando recuperación documental.
No se implementan Tools, MCP ni agentes como herramientas: solo se diseñan.

## 1. Validación del Laboratorio 2 y entorno
La configuración se lee del notebook anterior sin ejecutar sus celdas. Se conservan
rol, objetivos, restricciones, formato, `InMemoryHistoryProvider` y
`CompactionProvider` con `SummarizationStrategy`. La KB incrustada y el ejemplo con
la respuesta de Platinum se sustituyen por contexto recuperado. El original no cambia.

Seleccione **Python (dsc .env-dsc)**. Requiere `pip install -r requirements.txt`,
sesión de Azure CLI (`az login`) con permisos asignados y `.env` local basado en `env`.
Solo se envían textos sintéticos a los dos deployments de Azure. No hay API keys.
La ejecución completa genera nuevas llamadas de consumo a Azure; la colección local
se reutiliza cuando no cambia su huella. Los resultados guardados permiten revisarla
sin volver a ejecutar.


In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'laboratorio-02.ipynb').exists():
    ROOT = ROOT.parent
if not (ROOT / 'laboratorio-02.ipynb').exists():
    raise RuntimeError('Abra este notebook dentro del repositorio dsc.')
sys.path.insert(0, str(ROOT))
from IPython.display import display, Markdown
from lab03.rag import preparar_chunks, RepositorioRAG, guardar_json, OUTPUT
from lab03.agent import crear_clientes, crear_agente, configuracion_lab2
from lab03.pruebas import ejecutar_pruebas

credential, cliente_llm, cliente_embeddings, endpoint, modelo_embedding = crear_clientes()
agente, procedencia = crear_agente(cliente_llm, target_count=4, threshold=2)
sesion = agente.create_session()
guardar_json('procedencia_lab2.json', procedencia)
print('Agente:', procedencia['agente'])
print('Origen:', procedencia['notebook_origen'], '| SHA-256:', procedencia['sha256_origen'])
print('Autenticación: Microsoft Entra ID / DefaultAzureCredential')
print('Modelos: gpt-5.6-luna /', modelo_embedding)
print('Compactación de demostración: conserva 4 mensajes recientes; margen 2.')


Agente: CopilotoAsesorBancAnalytics
Origen: laboratorio-02.ipynb | SHA-256: 43839c1be70aad7d2ba9a1de256101d67da5cfb072dd832e3f9f18cfa4a0ce6c
Autenticación: Microsoft Entra ID / DefaultAzureCredential
Modelos: gpt-5.6-luna / text-embedding-3-small
Compactación de demostración: conserva 4 mensajes recientes; margen 2.


## 2. Conocimiento y estrategia de chunking
Reutilizamos el JSON extraído en el Lab 1: `outputs/bancanalytics_kb_index.json`.
Son tres PDF sintéticos: información institucional, tarjetas y Crédito Efectivo.
No mezclamos las condiciones de la guía Nova Andino de `lab01` con BancAnalytics.

**Estrategia:** división por estructura: una sección del índice = un chunk, con
título original como contexto. Límite de validación: **1,800 caracteres de contenido**
por sección; sin solapamiento. Ninguna sección actual supera el límite. Las tablas
se conservan completas para no separar ingresos, tasas y variantes. Si una sección
futura lo supera, el proceso pide revisar la segmentación en vez de truncarla.

**¿Por qué esta estrategia?** Las secciones ya son unidades temáticas. Un corte
fijo rompería filas de tablas; el solapamiento duplicaría condiciones y ocuparía
contexto innecesario. El título y la procedencia hacen interpretable cada fragmento.


In [2]:
chunks = preparar_chunks()
guardar_json('chunks.json', chunks)
documentos = sorted({c['metadata']['fuente'] for c in chunks})
categorias = sorted({c['metadata']['categoria'] for c in chunks})
print('Documentos:', len(documentos), documentos)
print('Categorías:', categorias)
print('Chunks:', len(chunks), '| IDs únicos:', len({c['id'] for c in chunks}))
print('Longitud máxima con título:', max(len(c['text']) for c in chunks))
display(chunks[0])


Documentos: 3 ['01_BancAnalytics_Informacion_Institucional.pdf', '02_BancAnalytics_Tarjeta_de_Credito.pdf', '03_BancAnalytics_Credito_Efectivo.pdf']
Categorías: ['Crédito Efectivo', 'Institucional', 'Tarjeta de Crédito']
Chunks: 22 | IDs únicos: 22
Longitud máxima con título: 1487


{'id': 'bca-inst-01',
 'text': 'Información institucional - Conoce BancAnalytics\n\n# Conoce BancAnalytics\n\nInformación institucional para atención a clientes · Versión 1.0 · Noviembre 2026\n\n\n## ¿Quiénes somos?\n\nBancAnalytics S.A. es un banco peruano con sede en Lima, fundado en 2009. Ofrecemos productos de crédito, ahorro y\nmedios de pago para personas naturales, apoyados en analítica de datos para dar ofertas más simples, rápidas y\npersonalizadas.\n\nMisión: hacer que las finanzas de cada peruano sean más claras y accesibles. Visión: ser el banco digital más confiable del\nPerú al 2030.',
 'metadata': {'fuente': '01_BancAnalytics_Informacion_Institucional.pdf',
  'titulo': 'Información institucional - Conoce BancAnalytics',
  'categoria': 'Institucional',
  'tipo_documento': 'Información institucional',
  'numero_chunk': 1,
  'paginas': '1',
  'sintetico': True}}

## 3. Estructura de chunks y metadatos
| Campo | Uso |
|---|---|
| `id` | ID único de sección del Lab 1; trazabilidad y actualización. |
| `text` → document | Título y contenido original recuperable. |
| embedding | Vector de Azure, fuera de los metadatos. |
| `categoria` | Filtro: Institucional, Tarjeta de Crédito o Crédito Efectivo. |
| `fuente`, `titulo`, `paginas` | Contexto y cita de origen. Páginas como texto simple. |
| `numero_chunk` | Orden dentro del documento. |
| `tipo_documento`, `sintetico` | Descripción/control de procedencia; futuros filtros posibles. |

**¿Qué elegimos para filtrar y qué para contexto?** En las pruebas filtramos
`categoria`; mostramos `fuente`, `titulo`, páginas e ID como contexto y procedencia.
Todos los metadatos son escalares; no se guardan listas ni objetos anidados.
Chroma no requiere declarar campos filtrables: los filtros se aplican a metadata.


## 4. Embeddings con Microsoft Agent Framework y Azure
Usamos `OpenAIEmbeddingClient`, `DefaultAzureCredential` y el deployment
`text-embedding-3-small` en el mismo endpoint del LLM. Documentos y preguntas usan
el mismo cliente/modelo. No hay Sentence Transformers ni embeddings automáticos de Chroma.


In [3]:
prueba_embedding = await cliente_embeddings.get_embeddings(
    ['Consulta ficticia de prueba para BancAnalytics.']
)
dimension = len(prueba_embedding[0].vector)
assert dimension > 0
print('Dimensiones del vector numérico generado en Azure:', dimension)


Dimensiones del vector numérico generado en Azure: 1536


## 5. Persistencia y recuperación en ChromaDB
Base local en `chroma_proyecto/`, fuera de Git. Usamos `PersistentClient`, métrica
cosine, `embedding_function=None` y vectores explícitos de Azure. La huella de
contenido, estrategia, modelo y endpoint identifica la colección: una nueva versión
no se mezcla con registros obsoletos. `upsert` carga IDs únicos.

La distancia es **1 − similitud del coseno**: menor distancia significa mayor
similitud, pero no garantiza que el fragmento responda. No imponemos un umbral
arbitrario: el agente debe verificar que el contenido sustente el dato.


In [4]:
rag = RepositorioRAG(cliente_embeddings, modelo_embedding, endpoint)
manifiesto = await rag.indexar(chunks)
guardar_json('indice_manifest.json', manifiesto)
display(manifiesto)
assert manifiesto['chunks'] == len(chunks)
assert manifiesto['dimensiones'] == dimension


{'coleccion': 'bancanalytics_18144c1020684ea8a4d2',
 'huella': '18144c1020684ea8a4d2245bdc118dd289a13faa2143177bf4e7bade44a20d82',
 'modelo_embeddings': 'text-embedding-3-small',
 'dimensiones': 1536,
 'chunks': 22,
 'documentos': ['01_BancAnalytics_Informacion_Institucional.pdf',
  '02_BancAnalytics_Tarjeta_de_Credito.pdf',
  '03_BancAnalytics_Credito_Efectivo.pdf'],
 'categorias': ['Crédito Efectivo', 'Institucional', 'Tarjeta de Crédito'],
 'estrategia': 'secciones-v1-max1800-sin-solapamiento',
 'metrica': 'cosine'}

In [5]:
pregunta = '¿Qué requisitos tiene Crédito Efectivo?'
sin_filtro = await rag.recuperar(pregunta)
con_filtro = await rag.recuperar(pregunta, categoria='Crédito Efectivo')
guardar_json('comparacion_filtros.json', {'pregunta': pregunta, 'sin_filtro': sin_filtro, 'con_filtro': con_filtro})
for nombre, fragmentos in [('SIN FILTRO', sin_filtro), ('FILTRO: Crédito Efectivo', con_filtro)]:
    print(nombre)
    for f in fragmentos:
        print(f['id'], '|', f['metadata']['categoria'], '|', f['metadata']['fuente'], '| distancia:', round(f['distancia'], 4))
assert con_filtro and all(f['metadata']['categoria'] == 'Crédito Efectivo' for f in con_filtro)


SIN FILTRO
bca-ce-06 | Crédito Efectivo | 03_BancAnalytics_Credito_Efectivo.pdf | distancia: 0.2559
bca-ce-08 | Crédito Efectivo | 03_BancAnalytics_Credito_Efectivo.pdf | distancia: 0.2928
bca-ce-04 | Crédito Efectivo | 03_BancAnalytics_Credito_Efectivo.pdf | distancia: 0.3013
FILTRO: Crédito Efectivo
bca-ce-06 | Crédito Efectivo | 03_BancAnalytics_Credito_Efectivo.pdf | distancia: 0.2559
bca-ce-08 | Crédito Efectivo | 03_BancAnalytics_Credito_Efectivo.pdf | distancia: 0.2928
bca-ce-04 | Crédito Efectivo | 03_BancAnalytics_Credito_Efectivo.pdf | distancia: 0.3013


## 6. Integración con el mismo agente
Flujo: **pregunta → embedding de Azure → ChromaDB → contexto con citas →
CopilotoAsesorBancAnalytics → respuesta**. La recuperación es una función del flujo,
no una Tool. `preguntar_con_rag` pasa la **misma sesión** a `agente.run`.
La KB ya no se inyecta completa en el prompt; solo los tres fragmentos recuperados.
El historial recuerda el perfil ficticio del asesor; las condiciones de productos
deben sustentarse en el contexto de la consulta actual. El prompt trata los documentos
como datos y exige reconocer información ausente.


## 7. Cinco pruebas de RAG y regresión del Lab 2
Se registra primero un perfil ficticio; después se ejecutan las cinco pruebas y
se vuelve a pedir el perfil tras la compactación. Se comprueba también el rechazo
de fútbol y la derivación de hipotecas. Todo ocurre en la **misma sesión**.

Evaluaciones: `correcto`, `parcial` o `incorrecto`, con comprobaciones y justificación.
Las reglas verifican datos esperados, cita, chunk y filtro; **no sustituyen una
revisión humana** del texto completo. Las distancias se guardan para inspección.
La compactación se acredita con mensajes excluidos y resúmenes reales del framework,
no únicamente con el número de turnos.


In [6]:
resultados, conversacion, resumen = await ejecutar_pruebas(agente, rag, sesion=sesion)
for caso in resultados:
    display(Markdown(f"### {caso['id']} · {caso['tipo']} · {caso['evaluacion']}\n"
                     f"**Pregunta:** {caso['pregunta']}\n\n"
                     f"**Filtro:** {caso['filtro']} · **Esperado:** {caso['esperado']}\n\n"
                     f"{caso['respuesta']}\n\n**Evaluación:** {caso['justificacion']}"))
    display([{'id': f['id'], 'fuente': f['metadata']['fuente'], 'distancia': f['distancia']} for f in caso['fragmentos']])
display(resumen)


RAG-01 | correcto | chunks: ['bca-tc-03', 'bca-tc-06', 'bca-tc-05'] | memoria: {'almacenados': 4, 'excluidos': 0, 'resumenes': 0}


RAG-02 | correcto | chunks: ['bca-tc-03', 'bca-tc-06', 'bca-tc-05'] | memoria: {'almacenados': 6, 'excluidos': 0, 'resumenes': 0}


RAG-03 | correcto | chunks: ['bca-inst-03', 'bca-inst-05', 'bca-inst-07'] | memoria: {'almacenados': 8, 'excluidos': 0, 'resumenes': 0}


RAG-04 | correcto | chunks: ['bca-ce-06', 'bca-ce-08', 'bca-ce-04'] | memoria: {'almacenados': 11, 'excluidos': 4, 'resumenes': 1}


RAG-05 | correcto | chunks: ['bca-tc-03', 'bca-tc-06', 'bca-tc-04'] | memoria: {'almacenados': 14, 'excluidos': 7, 'resumenes': 2}


### RAG-01 · Pregunta directa · correcto
**Pregunta:** ¿Cuál es el ingreso mínimo mensual de la tarjeta Platinum?

**Filtro:** None · **Esperado:** S/ 6,000 mensuales, con cita de la tabla de variantes.

ALCANCE: EN_ALCANCE

### 1) Resumen de la consulta
El asesor consulta el ingreso mínimo mensual requerido para la Tarjeta de Crédito Platinum.

### 2) Respuesta para el asesor
El ingreso mínimo mensual para la Tarjeta de Crédito **Platinum es S/ 6,000**.

### 3) Mensaje sugerido para el cliente
“Para acceder a la Tarjeta de Crédito Platinum, el ingreso mínimo mensual requerido es de S/ 6,000, sujeto a evaluación crediticia y demás requisitos aplicables.”

### 4) Oportunidades de oferta
Ninguna.

### 5) Fuentes
- [bca-tc-03 | 02_BancAnalytics_Tarjeta_de_Credito.pdf | pág. 1]

**Evaluación:** Cumple todas las comprobaciones explícitas del caso.

... (6 outputs left out)

[{'id': 'bca-ce-06',
  'fuente': '03_BancAnalytics_Credito_Efectivo.pdf',
  'distancia': 0.24507135152816772},
 {'id': 'bca-ce-08',
  'fuente': '03_BancAnalytics_Credito_Efectivo.pdf',
  'distancia': 0.3009611964225769},
 {'id': 'bca-ce-04',
  'fuente': '03_BancAnalytics_Credito_Efectivo.pdf',
  'distancia': 0.32925617694854736}]

### RAG-05 · Información ausente · correcto
**Pregunta:** ¿Cuál es el monto exacto en dólares de la cobertura del seguro de viaje de Platinum?

**Filtro:** Tarjeta de Crédito · **Esperado:** Reconocer que la cobertura exacta no está documentada, sin inventar una cifra.

ALCANCE: EN_ALCANCE

### 1) Resumen de la consulta
El asesor consulta el monto exacto en dólares de la cobertura del seguro de viaje de la Tarjeta Platinum.

### 2) Respuesta para el asesor
El contexto indica que el seguro de viaje está **incluido** en la Tarjeta Platinum, pero **no informa el monto de cobertura en dólares**. No dispongo de información suficiente para precisar esa cifra.

### 3) Mensaje sugerido para el cliente
“La Tarjeta Platinum incluye un seguro de viaje. El monto exacto de cobertura debe verificarse en las condiciones o documentación específica del seguro.”

### 4) Oportunidades de oferta
Ninguna.

### 5) Fuentes
- [bca-tc-03 | 02_BancAnalytics_Tarjeta_de_Credito.pdf | pág. 1]

**Evaluación:** Cumple todas las comprobaciones explícitas del caso.

[{'id': 'bca-tc-03',
  'fuente': '02_BancAnalytics_Tarjeta_de_Credito.pdf',
  'distancia': 0.5290632843971252},
 {'id': 'bca-tc-06',
  'fuente': '02_BancAnalytics_Tarjeta_de_Credito.pdf',
  'distancia': 0.6402327418327332},
 {'id': 'bca-tc-04',
  'fuente': '02_BancAnalytics_Tarjeta_de_Credito.pdf',
  'distancia': 0.6864268779754639}]

{'fecha_utc': '2026-11-26T03:50:17.909542+00:00',
 'estado': 'ejecutado_en_azure',
 'llm': 'gpt-5.6-luna',
 'embeddings': 'text-embedding-3-small',
 'autenticacion': 'DefaultAzureCredential / Microsoft Entra ID; sin API keys',
 'pruebas_rag': {'RAG-01': 'correcto',
  'RAG-02': 'correcto',
  'RAG-03': 'correcto',
  'RAG-04': 'correcto',
  'RAG-05': 'correcto'},
 'memoria_y_compactacion': {'ingreso': True,
  'monto': True,
  'tipo_preaprobada': True,
  'tea': True,
  'vigencia': True,
  'compactacion_real': True},
 'alcance': {'fuera_de_alcance': True, 'otra_especialidad': True},
 'misma_sesion': True,
 'criterio_aprobacion': 'Comprobaciones explícitas; no sustituye revisión humana de evidencia.',
 'versiones': {'agent-framework-core': '1.26.0',
  'agent-framework-openai': '1.18.2',
  'chromadb': '1.5.9',
  'azure-identity': '1.27.0'},
 'comprobaciones_superadas': True}

In [7]:
memoria = next(c for c in conversacion if c['tipo'] == 'memoria_tras_compactacion')
display(Markdown('### Perfil recordado tras compactación\n' + memoria['respuesta']))
display(memoria['memoria'])
display(memoria['comprobaciones'])


### Perfil recordado tras compactación
ALCANCE: EN_ALCANCE

### 1) Resumen de la consulta
Se solicita recordar únicamente los datos de la oferta del perfil ficticio compartido al inicio.

### 2) Respuesta para el asesor
- **Ingreso neto:** S/ 7,200 mensuales.
- **Monto:** oferta **pre-aprobada de hasta S/ 35,000**.
- **Tipo de oferta:** pre-aprobada, sujeta a validación; no equivale a una aprobación ni garantiza el desembolso.
- **TEA:** 18.9%.
- **Vigencia:** 10 días.

### 3) Mensaje sugerido para el cliente
“Actualmente cuentas con una oferta pre-aprobada de hasta S/ 35,000, con una TEA de 18.9%, vigente por 10 días y sujeta a validación. Esta oferta no garantiza la aprobación ni el desembolso.”

### 4) Oportunidades de oferta
Considerar el Crédito Efectivo para la remodelación de la casa, verificando previamente la capacidad de pago y evitando que las cuotas totales superen referencialmente el 40% del ingreso neto.

### 5) Fuentes
- Perfil ficticio compartido por el asesor al inicio de la conversación.

{'almacenados': 17, 'excluidos': 10, 'resumenes': 3}

{'ingreso': True,
 'monto': True,
 'tipo_preaprobada': True,
 'tea': True,
 'vigencia': True,
 'compactacion_real': True}

### Reflexión: ¿dónde funcionó y qué dificultades observamos?
Las pruebas distinguen recuperación y respuesta: encontrar un chunk no demuestra
que el dato esté presente. La consulta de cobertura de seguro es deliberadamente
difícil: la tabla indica que existe seguro de viaje, pero no su suma asegurada.
El agente debe reconocer esa ausencia aunque recupere el documento correcto.

Una sección de exoneración de membresía contiene texto extraído defectuoso; no lo
corregimos inventando cifras y no lo usamos como evidencia de condiciones exactas.
La tabla completa preserva relaciones, pero limita la precisión por fila; en una
colección más grande convendría normalizar tablas y validar un conjunto mayor de
preguntas. Este laboratorio pequeño no estima una precisión general de producción.
El resultado observado de cada caso y sus posibles dificultades aparece abajo.


In [8]:
for caso in resultados:
    print(caso['id'], caso['evaluacion'], '—', caso['justificacion'])
print('Memoria y compactación:', resumen['memoria_y_compactacion'])
print('Alcance:', resumen['alcance'])
print('Comprobaciones superadas:', resumen['comprobaciones_superadas'])
print('Evidencia completa en:', OUTPUT.relative_to(ROOT))


RAG-01 correcto — Cumple todas las comprobaciones explícitas del caso.
RAG-02 correcto — Cumple todas las comprobaciones explícitas del caso.
RAG-03 correcto — Cumple todas las comprobaciones explícitas del caso.
RAG-04 correcto — Cumple todas las comprobaciones explícitas del caso.
RAG-05 correcto — Cumple todas las comprobaciones explícitas del caso.
Memoria y compactación: {'ingreso': True, 'monto': True, 'tipo_preaprobada': True, 'tea': True, 'vigencia': True, 'compactacion_real': True}
Alcance: {'fuera_de_alcance': True, 'otra_especialidad': True}
Comprobaciones superadas: True
Evidencia completa en: lab03\outputs


## 8. Diseño de Tools y evaluación de Agent as a Tool
**Solo planificación; no implementadas en este laboratorio.**

| Nombre | Propósito/cuándo | Entradas | Salidas | Fuente prevista | Justificación |
|---|---|---|---|---|---|
| `buscar_conocimiento` | Consultar condiciones documentadas; elegir búsqueda cuando sea necesaria. | pregunta: str, categoría: str opcional, top_k: int 1–5 | chunks, distancia, ID, PDF/página | Chroma local con documentos sintéticos | Recuperación comprobable y filtros; el LLM no debe memorizar condiciones. |
| `consultar_oferta_demo` | Ver perfil/oferta ficticia vigente antes de orientar al asesor. | cliente_id: str con prefijo CLI-DEMO | encontrado: bool, perfil, tipo de oferta, monto, TEA, vigencia o error | JSON local 100 % sintético | No inventar ofertas ni convertir una pre-aprobada en aprobada; validar ID y caso ausente. |
| `simular_cuota_referencial` | Calcular cuota del sistema francés a pedido del asesor. | monto: número positivo, tea_pct: número no negativo, meses: entero positivo | cuota, total, interés, supuestos o error | Fórmula local con parámetros ficticios | Cálculo determinista, validación de plazo/tasa y tasa cero; excluye seguros y comisiones, no aprueba créditos. |

**¿Hace falta Agent as a Tool?** No por ahora: un único copiloto cubre tarjetas,
Crédito Efectivo y consultas institucionales. Recuperación y cálculo son funciones
acotadas; otro LLM añadiría latencia sin una responsabilidad necesaria distinta.
Las hipotecas se derivan verbalmente a otra especialidad, sin simular una llamada.
Si se amplía el alcance, un especialista hipotecario recibiría únicamente una
consulta/perfil ficticio mínimo y devolvería orientación documentada con citas y
límites, nunca una aprobación. Habría que evaluar utilidad antes de implementarlo.

## 9. Entregables y reproducción
- Notebook principal `laboratorio-03.ipynb` con salidas reales de Azure.
- Módulos `lab03/rag.py`, `lab03/agent.py`, `lab03/pruebas.py`.
- Evidencias JSON en `lab03/outputs/`: chunks, manifiesto, consultas con/sin filtro,
  cinco pruebas, conversación, historial compactado y resumen de ejecución.
- Pruebas locales: `python -m unittest lab03.test_rag -v` (fixture explícita; no Azure).
- Ejecución completa: `python scripts/ejecutar_lab03.py`.

La guía solicita un PR **no borrador** de `l3_DSC_30ME` hacia `main` en
`dmaravi-ds/dsc`, abierto desde la cuenta del participante.

Referencias técnicas: [Microsoft Agent Framework](https://learn.microsoft.com/en-us/agent-framework/),
[cliente OpenAI](https://github.com/microsoft/agent-framework/tree/main/python/packages/openai),
[colecciones Chroma](https://docs.trychroma.com/docs/collections/manage-collections).


In [9]:
credential.close()
assert resumen['comprobaciones_superadas'], 'Hay comprobaciones pendientes: revise pruebas_rag.json y conversacion.json.'
print('Laboratorio 3: ejecución completa y comprobaciones explícitas superadas.')


Laboratorio 3: ejecución completa y comprobaciones explícitas superadas.
